# 04 — Econometric Analysis

## Objective

Estimate the relationship between climate conditions and tourism activity using panel data models.

---

## Data

- Final dataset (Notebook 03)
- NUTS2 regions (2010–2023)

---

## Method

- Log transformation of key variables  
- OLS with time fixed effects  
- Panel fixed effects (region + year)  
- Non-linear specification (temperature squared)  

---

## Output

- Estimated coefficients  
- Statistical significance  
- Non-linear effects (optimal temperature)  

---

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv("../data/processed/final_dataset_nuts2_2010_2023.csv")

df = df[
    (df["tourism_nights"] > 0) &
    (df["gdp"] > 0) &
    (df["population"] > 0)
].copy()

df["log_tourism"] = np.log(df["tourism_nights"])
df["log_gdp"] = np.log(df["gdp"])
df["log_pop"] = np.log(df["population"])

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[
    "log_tourism","log_gdp","log_pop",
    "temperature","precipitation"
])

In [2]:
print("\n=== SUMMARY STATS ===")
print(df[[
    "tourism_nights",
    "gdp",
    "population",
    "temperature",
    "precipitation"
]].describe().T)


=== SUMMARY STATS ===
                 count          mean           std           min  \
tourism_nights  3274.0  3.259988e+06  4.170396e+06  29075.000000   
gdp             3274.0  5.468788e+04  6.966256e+04   1156.530000   
population      3274.0  1.952058e+06  1.762040e+06  27734.000000   
temperature     3274.0  1.092456e+01  3.334373e+00     -1.040769   
precipitation   3274.0  1.092658e+03  1.813768e+03     16.440392   

                          25%           50%           75%           max  
tourism_nights  922548.750000  1.930944e+06  3.927160e+06  4.944098e+07  
gdp              16192.730000  3.295590e+04  6.657937e+04  8.296383e+05  
population      964209.250000  1.488696e+06  2.331118e+06  1.590795e+07  
temperature          9.216048  1.079668e+01  1.266323e+01  2.103299e+01  
precipitation      362.392185  6.888876e+02  1.191565e+03  1.982463e+04  


In [3]:
def variation(x):
    return pd.Series({
        "overall_std": x.std(),
        "between_std": x.groupby(df["region"]).mean().std(),
        "within_std": (x - x.groupby(df["region"]).transform("mean")).std()
    })

for v in ["tourism_nights","gdp","temperature"]:
    print(f"\n{v}")
    print(variation(df[v]))


tourism_nights
overall_std    4.170396e+06
between_std    3.635730e+06
within_std     1.808820e+06
dtype: float64

gdp
overall_std    69662.561278
between_std    67130.976893
within_std     10964.125666
dtype: float64

temperature
overall_std    3.334373
between_std    3.284538
within_std     0.650638
dtype: float64


In [4]:
model = smf.ols(
    "log_tourism ~ log_gdp + log_pop + temperature + precipitation + C(year)",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["region"]})

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            log_tourism   R-squared:                       0.528
Model:                            OLS   Adj. R-squared:                  0.525
Method:                 Least Squares   F-statistic:                     122.8
Date:                Sun, 03 May 2026   Prob (F-statistic):          1.62e-116
Time:                        16:54:31   Log-Likelihood:                -3755.1
No. Observations:                3274   AIC:                             7546.
Df Residuals:                    3256   BIC:                             7656.
Df Model:                          17                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             4.5628      0.86

In [5]:
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

df = df.reset_index()
df = df.set_index(["region", "year"]).sort_index()

df["temp_anomaly"] = df["temperature"] - df.groupby(level=0)["temperature"].transform("mean")

X = sm.add_constant(df[["temp_anomaly", "precipitation"]])

model_fe = PanelOLS(
    df["log_tourism"],
    X,
    entity_effects=True,
    time_effects=True
).fit(cov_type="clustered", cluster_entity=True)

print(model_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:            log_tourism   R-squared:                        0.0236
Estimator:                   PanelOLS   R-squared (Between):              0.0038
No. Observations:                3274   R-squared (Within):               0.1328
Date:                Sun, May 03 2026   R-squared (Overall):              0.0304
Time:                        16:54:34   Log-likelihood                   -1693.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      36.174
Entities:                         272   P-value                           0.0000
Avg Obs:                       12.037   Distribution:                  F(2,2987)
Min Obs:                       2.0000                                           
Max Obs:                       14.000   F-statistic (robust):             9.4002
                            

In [6]:
df["temp_sq"] = df["temp_anomaly"]**2

X = sm.add_constant(df[["temp_anomaly","temp_sq","precipitation"]])

fe_nl = PanelOLS(
    df["log_tourism"],
    X,
    entity_effects=True,
    time_effects=True
).fit(cov_type="clustered", cluster_entity=True)

print(fe_nl.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:            log_tourism   R-squared:                        0.0398
Estimator:                   PanelOLS   R-squared (Between):             -0.0038
No. Observations:                3274   R-squared (Within):               0.1691
Date:                Sun, May 03 2026   R-squared (Overall):              0.0341
Time:                        16:54:34   Log-likelihood                   -1666.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      41.269
Entities:                         272   P-value                           0.0000
Avg Obs:                       12.037   Distribution:                  F(3,2986)
Min Obs:                       2.0000                                           
Max Obs:                       14.000   F-statistic (robust):             6.8196
                            

In [7]:
beta1 = fe_nl.params["temp_anomaly"]
beta2 = fe_nl.params["temp_sq"]

T_star = -beta1 / (2 * beta2)

mean_temp = df["temperature"].mean()
optimal_temp = mean_temp + T_star

print("Optimal temperature anomaly:", T_star)
print("Approx optimal temperature (°C):", optimal_temp)

Optimal temperature anomaly: 0.5581398241934623
Approx optimal temperature (°C): 11.482696308039523
